In [1]:
from google.colab import drive
drive.mount('/content/drive')

!cp /content/drive/MyDrive/NN_project/data.zip /content/
!cp /content/drive/MyDrive/NN_project/src.zip /content/
!cp /content/drive/MyDrive/NN_project/scripts.zip /content/

!unzip -q /content/data.zip -d /content/
!unzip -q /content/src.zip -d /content/
!unzip -q /content/scripts.zip -d /content/



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
replace /content/__MACOSX/._data? [y]es, [n]o, [A]ll, [N]one, [r]ename: A
A
A
A
A
A
replace /content/__MACOSX/._src? [y]es, [n]o, [A]ll, [N]one, [r]ename: A
replace /content/__MACOSX/._scripts? [y]es, [n]o, [A]ll, [N]one, [r]ename: A


In [2]:
import sys
import os
import torch
import numpy as np
from torch.utils.data import DataLoader, Subset
from tqdm import tqdm
import torchvision
from torchvision.transforms import v2

sys.path.append('/content/')

from src.datasets.dataset import PKLotDataset, collate_fn
from src.models.custom_detector import get_custom_detector

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Używane urządzenie: {device}")

def get_transform():
    return v2.Compose([
        v2.ToImage(),
        # Zmniejszamy zdjęcia do połowy rozmiaru
        v2.Resize(size=(360, 640), antialias=True),
        v2.ToDtype(torch.float32, scale=True),
    ])

full_dataset = PKLotDataset(
    root_dir="/content/data/raw/train",
    annotation_file="/content/data/raw/train/_annotations.coco.json",
    transforms=get_transform()
)

subset_size = min(1000, len(full_dataset))
indices = np.random.choice(len(full_dataset), subset_size, replace=False)
train_dataset = Subset(full_dataset, indices)

train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True,
    num_workers=2,
    collate_fn=collate_fn
)

model = get_custom_detector(num_classes=3)
model.to(device)

params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(params, lr=0.0005, weight_decay=0.0005)
lr_scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.1)

num_epochs = 10

print("Rozpoczynam trening...")
for epoch in range(num_epochs):
    model.train()
    epoch_loss = 0

    progress_bar = tqdm(train_loader, desc=f"Epoka {epoch+1}/{num_epochs}")

    for images, targets in progress_bar:
        images = list(image.to(device) for image in images)

        fixed_targets = []
        for t in targets:
            t_dict = {}
            for k, v in t.items():
                if k == "boxes" and v.numel() == 0:
                    t_dict[k] = torch.empty((0, 4), dtype=torch.float32, device=device)
                else:
                    t_dict[k] = v.to(device)
            fixed_targets.append(t_dict)
        targets = fixed_targets

        loss_dict = model(images, targets)
        losses = sum(loss for loss in loss_dict.values())

        optimizer.zero_grad()
        losses.backward()
        optimizer.step()

        epoch_loss += losses.item()
        progress_bar.set_postfix({'loss': losses.item()})

    lr_scheduler.step()
    print(f"Średnia strata w epoce {epoch+1}: {epoch_loss/len(train_loader):.4f}")

os.makedirs('/content/checkpoints', exist_ok=True)
torch.save(model.state_dict(), '/content/checkpoints/custom_model_subset.pth')

!cp /content/checkpoints/custom_model_subset.pth /content/drive/MyDrive/

Używane urządzenie: cuda
Rozpoczynam trening...


Epoka 1/10: 100%|██████████| 63/63 [01:32<00:00,  1.47s/it, loss=17.1]


Średnia strata w epoce 1: 18.7343


Epoka 2/10: 100%|██████████| 63/63 [01:33<00:00,  1.48s/it, loss=16]


Średnia strata w epoce 2: 16.3876


...

Średnia strata w epoce 8: 10.1483


Epoka 9/10: 100%|██████████| 63/63 [01:35<00:00,  1.52s/it, loss=10.5]


Średnia strata w epoce 9: 9.8951


Epoka 10/10: 100%|██████████| 63/63 [01:36<00:00,  1.53s/it, loss=11.1]


Średnia strata w epoce 10: 9.8126


In [3]:
!pip install -q torchmetrics

import sys
import torch
from torch.utils.data import DataLoader
from tqdm import tqdm
from torchvision.transforms import v2
from torchmetrics.detection.mean_ap import MeanAveragePrecision

sys.path.append('/content/')
from src.datasets.dataset import PKLotDataset, collate_fn
from src.models.custom_detector import get_custom_detector

def evaluate_model():
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Używane urządzenie: {device}")

    transform = v2.Compose([
        v2.ToImage(),
        v2.Resize(size=(360, 640), antialias=True),
        v2.ToDtype(torch.float32, scale=True),
    ])

    test_dataset = PKLotDataset(
        root_dir="/content/data/raw/test",
        annotation_file="/content/data/raw/test/_annotations.coco.json",
        transforms=transform
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=32,
        shuffle=False,
        num_workers=2,
        collate_fn=collate_fn
    )

    model = get_custom_detector(num_classes=3)
    model.load_state_dict(torch.load('/content/drive/MyDrive/custom_model_subset.pth', map_location=device, weights_only=True))
    model.to(device)

    model.eval()

    metric = MeanAveragePrecision(box_format='xyxy', class_metrics=True)

    print(f"Rozpoczynam ewaluację")

    with torch.no_grad():
        progress_bar = tqdm(test_loader, desc="Ewaluacja")

        for images, targets in progress_bar:
            images = list(image.to(device) for image in images)

            fixed_targets = []
            for t in targets:
                t_dict = {}
                for k, v in t.items():
                    if k == "boxes" and v.numel() == 0:
                        t_dict[k] = torch.empty((0, 4), dtype=torch.float32, device=device)
                    else:
                        t_dict[k] = v.to(device)
                fixed_targets.append(t_dict)

            predictions = model(images)

            metric.update(predictions, fixed_targets)

    results = metric.compute()

    print("-" * 50)
    print("WYNIKI EWALUACJI:")
    print("-" * 50)
    print(f"mAP@50-95:    {results['map'].item():.4f}")
    print(f"mAP@50:       {results['map_50'].item():.4f}")
    print(f"mAP@75:       {results['map_75'].item():.4f}")
    print(f"Recall (MAR): {results['mar_100'].item():.4f}")
    print("-" * 50)

evaluate_model()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 46.5 MB/s eta 0:00:00
Używane urządzenie: cuda
Rozpoczynam ewaluację


Ewaluacja: 100%|██████████| 39/39 [00:41<00:00,  1.06s/it]


--------------------------------------------------
WYNIKI EWALUACJI:
--------------------------------------------------
mAP@50-95:    0.0501
mAP@50:       0.1340
mAP@75:       0.0270
Recall (MAR): 0.0913
--------------------------------------------------
